# 01 数据：建库、导入与更新

这个 notebook 只负责一件事：把研究需要的原始数据放进 PostgreSQL 数据库 `hindenburg`。
后面的 `02_回测.ipynb` 和 `03_展示.ipynb` 都只从数据库读数据。

| 表 | 内容 |
|---|---|
| `stock_close_adj` | 个股后复权收盘价（日期 × 代码） |
| `stock_state` | 每日交易状态：是否成交、上市天数、ST 等 |
| `index_close` / `index_meta` | 基准指数收盘价；`head_floor` = 数据源能给到的最早日期 |
| `index_constituents` | 指数成分股 |
| `stock_close_unadj` | 个股不复权收盘价（只给「不复权」对照实验用） |
| `jq_securities` | 聚宽证券基础信息缓存（区分新股 / 退市股） |

运行顺序：**第 1 节建表 → 第 2 节导入旧文件（只在表为空时执行一次）→ 第 3 节联网增量更新**。
整本从头运行即可，已有的数据不会重复导入。

## 0. 设置

- 数据库地址用环境变量 `HO_DB_URL`，默认连本机的 `hindenburg` 库（本机连接不需要密码）。
- 聚宽账号只用环境变量 `JQ_USER` / `JQ_PASSWORD` 提供，不要写进 notebook。
- `HO_LEGACY_DIR`：旧版的 `.pkl` / `.parquet` 数据所在目录，只在第一次导入时用到。

In [1]:
import io
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import sqlalchemy as sa

DB_URL = os.getenv('HO_DB_URL', 'postgresql+psycopg://localhost/hindenburg')
LEGACY_DIR = Path(os.getenv('HO_LEGACY_DIR', 'data'))              # 旧版 pkl 数据（递归查找）
LEGACY_INDEX_DIR = Path(os.getenv('HO_LEGACY_INDEX_DIR', 'index_store'))  # 旧版指数 parquet
OFFLINE = os.getenv('HO_OFFLINE', '').lower() in ('1', 'true', 'yes')     # 强制不联网

# 要维护的基准指数及各自需要的最早日期，和 02_回测 里的策略一一对应
INDEX_START = {'000985.CSI': '2005-01-01', '000001.SH': '2000-01-01', '000300.SH': '2005-04-08',
               '000905.SH': '2007-01-15', '000852.SH': '2014-10-17'}
CONSTITUENT_INDEXES = ['000300.SH', '000905.SH', '000852.SH']   # 需要成分股名单的指数

# 聚宽额度保护（单位：行）
JQ_RESERVE = int(os.getenv('JQ_RESERVE', 50_000))                 # 至少给别人留这么多
JQ_MAX_ROWS = int(os.getenv('JQ_MAX_ROWS', 20_000))               # 单次运行更新指数的上限
JQ_MAX_ROWS_PANEL = int(os.getenv('JQ_MAX_ROWS_PANEL', 500_000))  # 单次运行更新面板的上限
JQ_DRY_RUN = os.getenv('JQ_DRY_RUN', '') == '1'                   # 只打印计划，不真拉

engine = sa.create_engine(DB_URL)

两个读写小工具：大表用 PostgreSQL 的 `COPY` 批量进出，比逐行插入快几十倍。

In [2]:
def copy_in(df: pd.DataFrame, table: str, chunk: int = 2_000_000):
    """把 DataFrame 按列名批量写入表（列名必须和表一致），空值写成 NULL。"""
    cols = ', '.join(df.columns)
    raw = engine.raw_connection()
    try:
        pg = raw.driver_connection
        with pg.cursor() as cur:
            with cur.copy(f"COPY {table} ({cols}) FROM STDIN WITH (FORMAT csv, NULL '')") as cp:
                for i in range(0, len(df), chunk):
                    cp.write(df.iloc[i:i + chunk].to_csv(index=False, header=False, na_rep=''))
        pg.commit()
    finally:
        raw.close()
    print(f'  {table}: 写入 {len(df):,} 行')


def read_sql(sql: str, **read_csv_kw) -> pd.DataFrame:
    """用 COPY 把查询结果读成 DataFrame，大表比 pd.read_sql 快很多。"""
    buf = io.BytesIO()
    raw = engine.raw_connection()
    try:
        with raw.driver_connection.cursor() as cur:
            with cur.copy(f'COPY ({sql}) TO STDOUT WITH (FORMAT csv, HEADER)') as cp:
                for block in cp:
                    buf.write(block)
    finally:
        raw.close()
    buf.seek(0)
    return pd.read_csv(buf, **read_csv_kw)


def execute(sql: str, **params):
    with engine.begin() as con:
        return con.execute(sa.text(sql), params)


def scalar(sql: str, **params):
    with engine.connect() as con:
        return con.execute(sa.text(sql), params).scalar()


def one(sql: str, **params):
    with engine.connect() as con:
        return con.execute(sa.text(sql), params).one()

## 1. 建表

In [3]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS stock_close_adj (
    trade_date DATE NOT NULL, code TEXT NOT NULL, close DOUBLE PRECISION,
    PRIMARY KEY (trade_date, code));
CREATE TABLE IF NOT EXISTS stock_close_unadj (
    trade_date DATE NOT NULL, code TEXT NOT NULL, close DOUBLE PRECISION,
    PRIMARY KEY (trade_date, code));
CREATE TABLE IF NOT EXISTS stock_state (
    trade_date DATE NOT NULL, code TEXT NOT NULL,
    trade_status TEXT,        -- 原数据的中文取值：'交易' / '停牌一天' / ...
    status_t SMALLINT,        -- 1 = 当日有成交
    maxupdown REAL,           -- 1 涨停 / -1 跌停 / 0
    isst SMALLINT, issts SMALLINT, issusp SMALLINT,
    lenlist INTEGER,          -- 上市自然日数
    PRIMARY KEY (trade_date, code));
CREATE TABLE IF NOT EXISTS index_close (
    index_code TEXT NOT NULL, trade_date DATE NOT NULL, close DOUBLE PRECISION,
    PRIMARY KEY (index_code, trade_date));
CREATE TABLE IF NOT EXISTS index_meta (
    index_code TEXT PRIMARY KEY, head_floor DATE, note TEXT);
CREATE TABLE IF NOT EXISTS index_constituents (
    index_code TEXT NOT NULL, trade_date DATE NOT NULL, code TEXT NOT NULL, weight DOUBLE PRECISION,
    PRIMARY KEY (index_code, trade_date, code));
CREATE TABLE IF NOT EXISTS jq_securities (
    code TEXT PRIMARY KEY, start_date DATE, end_date DATE, fetched_at TIMESTAMP);
"""
with engine.begin() as con:
    con.exec_driver_sql(SCHEMA)
print('表已就绪')

表已就绪


## 2. 一次性导入旧版文件

旧版项目把数据存在 `stock_price_adj_close.pkl`、`stock_state.pkl`、`index_constituents_*.pkl`、
`index_store/*.parquet` 等文件里。这一节把它们导入数据库，**表里已有数据就跳过**，所以重复运行没有副作用。
找不到旧文件也会跳过（比如在新电脑上直接用聚宽从头拉数据）。

In [4]:
def find_legacy(name: str):
    """在 LEGACY_DIR 下递归查找文件（旧版数据可能放在子目录里）。"""
    if not LEGACY_DIR.exists():
        return None
    hits = sorted(LEGACY_DIR.rglob(name))
    return hits[0] if hits else None


def table_empty(table: str) -> bool:
    return scalar(f'SELECT NOT EXISTS (SELECT 1 FROM {table} LIMIT 1)')


def wide_to_long(px: pd.DataFrame) -> pd.DataFrame:
    """日期 × 代码 的宽表 → (trade_date, code, close) 长表，只保留有值的格子。"""
    px = px.copy()
    px.index = pd.to_datetime(px.index)
    long = px.rename_axis(index='trade_date', columns='code').stack(future_stack=True).dropna()
    long = long.rename('close').reset_index()
    long['trade_date'] = long['trade_date'].dt.date
    return long


def import_close(table: str, filename: str):
    path = find_legacy(filename)
    if not table_empty(table) or path is None:
        print(f'{table}: 跳过（{"已有数据" if path else "未找到 " + filename}）')
        return
    print(f'{table}: 从 {path.name} 导入…')
    copy_in(wide_to_long(pd.read_pickle(path)), table)


import_close('stock_close_adj', 'stock_price_adj_close.pkl')
import_close('stock_close_unadj', 'unadj_close_*.pkl')

stock_close_adj: 跳过（未找到 stock_price_adj_close.pkl）
stock_close_unadj: 跳过（未找到 unadj_close_*.pkl）


In [5]:
path = find_legacy('stock_state.pkl')
if not table_empty('stock_state') or path is None:
    print(f'stock_state: 跳过（{"已有数据" if path else "未找到 stock_state.pkl"}）')
else:
    print(f'stock_state: 从 {path.name} 导入…')
    st = pd.read_pickle(path)
    st.index = st.index.set_names(['trade_date', 'code'])
    st = st.reset_index()
    st['trade_date'] = pd.to_datetime(st['trade_date']).dt.date
    copy_in(st[['trade_date', 'code', 'trade_status', 'status_t', 'maxupdown',
                'isst', 'issts', 'issusp', 'lenlist']], 'stock_state')
    del st

stock_state: 跳过（未找到 stock_state.pkl）


In [6]:
if table_empty('index_constituents'):
    files = sorted(LEGACY_DIR.rglob('index_constituents_*.pkl')) if LEGACY_DIR.exists() else []
    for f in files:
        raw = pd.read_pickle(f)
        if not len(raw):
            continue
        code = f.stem.replace('index_constituents_', '')
        code = code if '.' in code else code.replace('_', '.')          # 000300_SH → 000300.SH
        df = raw.copy()
        df.index = df.index.set_names(['trade_date', 'code'])
        df = df.reset_index()
        df.insert(0, 'index_code', code)
        df['trade_date'] = pd.to_datetime(df['trade_date']).dt.date
        print(f'index_constituents: {f.name}')
        copy_in(df[['index_code', 'trade_date', 'code', 'weight']], 'index_constituents')
else:
    print('index_constituents: 跳过（已有数据）')

index_constituents: 跳过（已有数据）


In [7]:
if table_empty('index_close') and LEGACY_INDEX_DIR.exists():
    for f in sorted(LEGACY_INDEX_DIR.glob('*.parquet')):
        code = f.stem.replace('_', '.')
        s = pd.read_parquet(f)['close']
        df = pd.DataFrame({'index_code': code, 'trade_date': pd.to_datetime(s.index).date, 'close': s.values})
        print(f'index_close: {f.name}')
        copy_in(df.dropna(), 'index_close')
        meta = LEGACY_INDEX_DIR / f'{f.stem}.meta.json'
        if meta.exists():
            m = json.loads(meta.read_text())
            if m.get('head_floor'):
                execute('INSERT INTO index_meta VALUES (:c, :d, :n) ON CONFLICT (index_code) DO UPDATE '
                        'SET head_floor = EXCLUDED.head_floor', c=code, d=m['head_floor'], n=m.get('note'))
else:
    print('index_close: 跳过（已有数据或未找到 index_store）')

index_close: 跳过（已有数据或未找到 index_store）


导入后的概况：

In [8]:
def overview() -> pd.DataFrame:
    rows = []
    for t, key in [('stock_close_adj', 'code'), ('stock_close_unadj', 'code'), ('stock_state', 'code'),
                   ('index_close', 'index_code'), ('index_constituents', 'index_code')]:
        n, lo, hi, k = one(f'SELECT count(*), min(trade_date), max(trade_date), count(DISTINCT {key}) FROM {t}')
        rows.append({'表': t, '行数': f'{n:,}', '起始': lo, '结束': hi, '代码数': k})
    return pd.DataFrame(rows)

overview()

,表,行数,起始,结束,代码数
0,stock_close_adj,"17,794,688",2000-01-04,2026-08-07,6106
1,stock_close_unadj,"16,111,553",2005-01-04,2026-08-07,5506
2,stock_state,"17,875,816",2000-01-04,2026-08-07,6108
3,index_close,"21,703",2005-01-04,2026-08-07,5
4,index_constituents,"7,284,239",2005-01-04,2026-07-31,6


## 3. 聚宽增量更新

聚宽账号可能是共用的，所以这里的设计目标是「尽量少用额度」：

- 每次拉取前先查剩余额度，扣完会低于 `JQ_RESERVE` 就拒绝；单次运行累计行数也有上限。
- 指数只补缺口；头部缺口拉不到就把最早可得日期记进 `index_meta.head_floor`，以后不再重试。
- `JQ_DRY_RUN=1` 只打印计划，不真拉。

### 3.1 认证、额度与代码转换

In [9]:
_jq = None          # 已认证的 jqdatasdk 模块，保证只认证一次
_used_rows = 0      # 本次运行累计消耗的行数（估算）


def jq_auth():
    """认证聚宽，只做一次（重复认证会挤掉自己的上一个会话）。"""
    global _jq
    if _jq is not None:
        try:
            if _jq.is_auth():
                return _jq
        except Exception:
            pass
    import jqdatasdk as jq
    try:
        authed = jq.is_auth()
    except Exception:
        authed = False
    if not authed:
        user, password = os.getenv('JQ_USER'), os.getenv('JQ_PASSWORD')
        if not user or not password:
            raise RuntimeError('缺少聚宽账号。请设置 JQ_USER / JQ_PASSWORD')
        jq.auth(str(user), str(password))
    _jq = jq
    print(f'[JQ] 认证成功。{quota_str()}')
    return _jq


def quota() -> dict:
    jq = _jq if _jq is not None else jq_auth()
    try:
        return dict(jq.get_query_count())
    except Exception as e:
        return {'total': None, 'spare': None, 'error': str(e)}


def quota_str() -> str:
    q = quota()
    if q.get('spare') is None:
        return f'额度查询失败 ({q.get("error")})'
    return f'今日额度剩余 {q["spare"]:,} / {q["total"]:,} ({q["spare"] / q["total"]:.0%})'


def est_rows(start, end, n_codes: int = 1) -> int:
    """按每年 245 个交易日估算行数。"""
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days
    return max(1, int(days / 365 * 245) + 5) * n_codes


def guard(need: int, cap=None):
    """拉取前检查：本次累计不超过 cap，且扣完后剩余额度不低于 JQ_RESERVE。"""
    cap = cap if cap is not None else JQ_MAX_ROWS
    if _used_rows + need > cap:
        raise RuntimeError(f'本次运行已估算消耗 {_used_rows:,} 行, 再拉 {need:,} 行会超过上限 {cap:,}。'
                           f'确认无误可调高对应的 JQ_MAX_ROWS* 环境变量。')
    spare = quota().get('spare')
    if spare is None:
        print('[JQ] ⚠ 无法查询额度, 按谨慎处理: 仍继续但请留意用量')
        return
    if spare - need < JQ_RESERVE:
        raise RuntimeError(f'剩余额度 {spare:,} 行, 本次需约 {need:,} 行, 扣完将低于给他人预留的 '
                           f'{JQ_RESERVE:,} 行下限, 已拒绝。今日请用 HO_OFFLINE=1 跑纯本地, 或明日再更新。')
    print(f'[JQ] 预估消耗 {need:,} 行, 剩余 {spare:,} 行, 通过额度检查')


_ALIAS = {'000001.SH': '000001.XSHG', '000985.CSI': '000985.XSHG', '000300.SH': '000300.XSHG',
          '000905.SH': '000905.XSHG', '000852.SH': '000852.XSHG', '000688.SH': '000688.XSHG',
          '399006.SZ': '399006.XSHE', '399001.SZ': '399001.XSHE', '399673.SZ': '399673.XSHE'}


def to_jq_code(code: str) -> str:
    """数据库代码 → 聚宽代码，例如 000985.CSI → 000985.XSHG，830799.BJ → 830799.BJSE。"""
    if code in _ALIAS:
        return _ALIAS[code]
    if code.endswith(('.XSHG', '.XSHE', '.BJSE')):
        return code
    num, _, suf = code.partition('.')
    suf = suf.upper()
    if suf in ('SH', 'SS'):
        return f'{num}.XSHG'
    if suf in ('SZ', 'SE'):
        return f'{num}.XSHE'
    if suf in ('BJ', 'BSE', 'BJSE', 'NQ'):
        return f'{num}.BJSE'
    # 无后缀：8/4/920 开头为北交所，399/395 为深交所，其余为上交所
    if num.startswith(('8', '4', '920')):
        return f'{num}.BJSE'
    return f'{num}.XSHE' if num.startswith(('399', '395')) else f'{num}.XSHG'


def from_jq_code(code: str) -> str:
    """聚宽代码 → 数据库代码（to_jq_code 的逆运算）。"""
    num, _, suf = str(code).partition('.')
    return {'XSHG': f'{num}.SH', 'XSHE': f'{num}.SZ', 'BJSE': f'{num}.BJ'}.get(suf.upper(), str(code))


def last_closed_trade_day(jq) -> pd.Timestamp:
    """最近一个已收盘的交易日（按聚宽交易日历；当天 15:30 前算前一个交易日）。"""
    today = pd.Timestamp.today().normalize()
    try:
        days = jq.get_trade_days(end_date=str(today.date()), count=10)
    except Exception:
        return today
    if len(days) == 0:
        return today
    days = [pd.Timestamp(d) for d in days]
    last = days[-1]
    now = pd.Timestamp.now()
    if last == today and now.hour * 60 + now.minute < 15 * 60 + 30:
        last = days[-2] if len(days) >= 2 else last
    return last

### 3.2 指数收盘价：只补缺口

In [10]:
def fetch_index(code: str, start: str, end: str) -> pd.Series:
    """从聚宽拉 [start, end] 的指数收盘价；没有数据时返回空序列。"""
    global _used_rows
    jq = jq_auth()
    guard(est_rows(start, end))

    def get(jq_code):
        global _used_rows
        if JQ_DRY_RUN:
            print(f'[JQ] DRY RUN: 本应拉取 {jq_code} {start} ~ {end}')
            return pd.DataFrame()
        df = jq.get_price(jq_code, start_date=start, end_date=end, frequency='daily',
                          fields=['close'], skip_paused=False, fq=None)
        _used_rows += len(df) if df is not None else 0
        return df

    jq_code = to_jq_code(code)
    df = get(jq_code)
    if (df is None or df.empty) and jq_code.endswith(('.XSHG', '.XSHE')):   # 交易所猜错时换一个再试
        alt = jq_code.replace('.XSHG', '.XSHE') if jq_code.endswith('.XSHG') else jq_code.replace('.XSHE', '.XSHG')
        df2 = get(alt)
        if df2 is not None and not df2.empty:
            df = df2
    if df is None or df.empty:
        return pd.Series(dtype=float)
    s = df['close'].dropna()      # 早于指数起点的区间，聚宽返回「日期齐全但值全 NaN」的表
    s.index = pd.to_datetime(s.index)
    return s[~s.index.duplicated(keep='last')].sort_index()


def update_index(code: str, start: str, end: str, fresh_days: int = 5):
    """把 code 的收盘价补到覆盖 [start, end]；头部缺口拉不到就记 head_floor，以后不再重试。"""
    lo, hi = one('SELECT min(trade_date), max(trade_date) FROM index_close WHERE index_code = :c', c=code)
    floor = scalar('SELECT head_floor FROM index_meta WHERE index_code = :c', c=code)
    req_start = pd.Timestamp(start)
    req_end = min(pd.Timestamp(end), pd.Timestamp.today().normalize())
    gaps = []
    if lo is None:
        gaps.append((req_start, req_end, 'full'))
    else:
        lo, hi = pd.Timestamp(lo), pd.Timestamp(hi)
        want_head = req_start if floor is None else max(req_start, pd.Timestamp(floor))
        if lo > want_head + pd.Timedelta(days=30):
            gaps.append((want_head, lo - pd.Timedelta(days=1), 'head'))
        if hi < req_end - pd.Timedelta(days=fresh_days):
            gaps.append((hi + pd.Timedelta(days=1), req_end, 'tail'))
    for gs, ge, kind in gaps:
        print(f'[index_close] {code} 补{ {"head": "头部", "tail": "尾部", "full": "全量"}[kind] } {gs:%Y-%m-%d} ~ {ge:%Y-%m-%d}')
        s = fetch_index(code, gs.strftime('%Y-%m-%d'), ge.strftime('%Y-%m-%d'))
        if s.empty:
            if kind in ('head', 'full') and lo is not None:
                execute('INSERT INTO index_meta VALUES (:c, :d, :n) ON CONFLICT (index_code) DO UPDATE '
                        'SET head_floor = EXCLUDED.head_floor, note = EXCLUDED.note',
                        c=code, d=lo.date(), n='头部缺口拉取为空, 已记录不再重试')
                print('  该段无数据, 已记录 head_floor, 后续不再重试')
            continue
        execute('DELETE FROM index_close WHERE index_code = :c AND trade_date BETWEEN :a AND :b',
                c=code, a=s.index.min().date(), b=s.index.max().date())
        copy_in(pd.DataFrame({'index_code': code, 'trade_date': s.index.date, 'close': s.values}), 'index_close')
    new_lo = scalar('SELECT min(trade_date) FROM index_close WHERE index_code = :c', c=code)
    if new_lo is not None and pd.Timestamp(new_lo) > req_start + pd.Timedelta(days=30):
        execute('INSERT INTO index_meta (index_code, head_floor) VALUES (:c, :d) ON CONFLICT (index_code) '
                'DO UPDATE SET head_floor = EXCLUDED.head_floor', c=code, d=new_lo)

### 3.3 成分股名单

聚宽的 `get_index_weights` 一次只能取一天的名单，所以按月末采样（成分股半年调一次，月频足够）。

In [11]:
def fetch_constituents(index_code: str, start: str, end: str, freq: str = 'ME'):
    jq = jq_auth()
    jq_code = to_jq_code(index_code)
    frames, missed = [], 0
    for d in pd.date_range(start, end, freq=freq):
        try:
            w = jq.get_index_weights(jq_code, date=str(d.date()))
        except Exception:
            missed += 1
            continue
        if w is None or len(w) == 0:
            missed += 1
            continue
        frames.append(pd.DataFrame({'index_code': index_code, 'trade_date': d.date(),
                                    'code': [from_jq_code(c) for c in w.index], 'weight': w['weight'].values}))
    if not frames:
        raise ValueError(f'{index_code} 没取到任何成分股, 检查代码或账号权限')
    out = pd.concat(frames, ignore_index=True)
    execute('DELETE FROM index_constituents WHERE index_code = :c', c=index_code)
    copy_in(out, 'index_constituents')
    print(f'{index_code}: {len(frames)} 个采样日' + (f' ({missed} 个日期无数据)' if missed else ''))

### 3.4 个股价格：按涨跌幅复利拼接

拼接方法：拉一段**后复权**价，只取其中的日涨跌幅，再从库里最后一天的价格往后复利滚算。
这样即使期间有分红送股（复权基准变了），接缝处也不会跳变。

- 不能用不复权价算涨跌幅：除权日的断崖下跌会被当成真实下跌，伪造出新低。
- 只更新库里已有的股票，之后新上市的股票不会自动加入（已知局限）。

In [12]:
BATCH_SIZE = 2500   # 单次 get_price 请求的股票数上限


def isolate_bad_codes(jq, jq_codes, probe_date, fields):
    """找出聚宽不认识的代码（批量请求里只要有一个无效代码就整批报错，所以用二分法隔离）。"""
    bad = []

    def check(codes):
        if not codes:
            return
        try:
            jq.get_price(codes, start_date=probe_date, end_date=probe_date, frequency='daily',
                         fields=fields, skip_paused=False, fq=None, panel=False)
        except Exception as e:
            if len(codes) == 1:
                bad.append((codes[0], str(e).strip()[:60]))
                return
            mid = len(codes) // 2
            check(codes[:mid])
            check(codes[mid:])

    for i in range(0, len(jq_codes), BATCH_SIZE):
        check(jq_codes[i:i + BATCH_SIZE])
    bad_set = {c for c, _ in bad}
    return [c for c in jq_codes if c not in bad_set], bad


def fetch_close_in_batches(jq, jq_codes, start, end, fq):
    """分批拉取收盘价宽表；某批失败时二分重试，只跳过真正出错的代码。"""
    parts, bad = [], []

    def pull(codes, depth=0):
        if not codes:
            return
        try:
            df = jq.get_price(codes, start_date=start, end_date=end, frequency='daily',
                              fields=['close'], fq=fq, panel=False, skip_paused=False)
            if df is not None and not df.empty:
                parts.append(df.pivot(index='time', columns='code', values='close').sort_index())
        except Exception as e:
            if len(codes) == 1:
                bad.append((codes[0], str(e).strip()[:60]))
                return
            if depth == 0:
                print(f'    ⚠ 该批出错, 二分隔离中: {str(e).strip()[:60]}')
            mid = len(codes) // 2
            pull(codes[:mid], depth + 1)
            pull(codes[mid:], depth + 1)

    batches = [jq_codes[i:i + BATCH_SIZE] for i in range(0, len(jq_codes), BATCH_SIZE)]
    for i, b in enumerate(batches, 1):
        print(f'  批次 {i}/{len(batches)} ({len(b)} 只)...')
        pull(b)
    if bad:
        print(f'  {len(bad)} 只代码拉取失败, 已跳过: {[c for c, _ in bad[:5]]}')
    return parts


def securities(jq) -> pd.DataFrame:
    """聚宽证券基础信息（上市 / 退市日期），库里缓存一天。"""
    age = scalar("SELECT now() - max(fetched_at) < interval '1 day' FROM jq_securities")
    if not age:
        info = jq.get_all_securities(types=['stock'])
        df = pd.DataFrame({'code': [str(i) for i in info.index], 'start_date': info['start_date'].values,
                           'end_date': info['end_date'].values, 'fetched_at': pd.Timestamp.now()})
        execute('DELETE FROM jq_securities')
        copy_in(df, 'jq_securities')
    return read_sql('SELECT code, start_date, end_date FROM jq_securities',
                    parse_dates=['start_date', 'end_date']).set_index('code')


def classify(info: pd.DataFrame, jq_codes, last_date) -> dict:
    """按上市 / 退市日期分类：normal / new_listing / delisted / unknown。"""
    out = {}
    for c in jq_codes:
        if c not in info.index:
            out[c] = 'unknown'
            continue
        row = info.loc[c]
        end_date = row['end_date'] if pd.notna(row['end_date']) else None
        if end_date is not None and end_date <= last_date:
            out[c] = 'delisted'
        elif row['start_date'] > last_date:
            out[c] = 'new_listing'
        else:
            out[c] = 'normal'
    return out


def update_price_panel(end=None, dry_run=False):
    last_date = pd.Timestamp(scalar('SELECT max(trade_date) FROM stock_close_adj'))
    anchor = read_sql(f"SELECT code, close FROM stock_close_adj WHERE trade_date = '{last_date.date()}'"
                      ).set_index('code')['close']
    codes = read_sql('SELECT DISTINCT code FROM stock_close_adj')['code'].tolist()
    jq = jq_auth()
    end_ts = pd.Timestamp(end) if end else last_closed_trade_day(jq)
    print(f'库中价格到 {last_date:%Y-%m-%d} → 目标 {end_ts:%Y-%m-%d}')
    if end_ts <= last_date:
        print('已是最新, 无需更新')
        return

    code_map = {c: to_jq_code(c) for c in codes}
    inv_map = {v: k for k, v in code_map.items()}
    cls = {inv_map[k]: v for k, v in classify(securities(jq), list(code_map.values()), last_date).items()}
    n = pd.Series(cls).value_counts().to_dict()
    print(f'正常 {n.get("normal", 0)} 只, 已退市 {n.get("delisted", 0)} 只, 聚宽无此代码 {n.get("unknown", 0)} 只')
    fetch_codes = [code_map[c] for c, v in cls.items() if v == 'normal']
    need = len(fetch_codes) * max(1, (end_ts - last_date).days * 5 // 7 + 3)
    print(f'待请求 {len(fetch_codes)} 只, 预估 {need:,} 行')
    if dry_run or not fetch_codes:
        return
    guard(need, cap=JQ_MAX_ROWS_PANEL)

    parts = fetch_close_in_batches(jq, fetch_codes, last_date.strftime('%Y-%m-%d'), end_ts.strftime('%Y-%m-%d'), fq='post')
    if not parts:
        print('未取到任何数据, 中止')
        return
    raw = pd.concat(parts, axis=1)
    raw.columns = [inv_map.get(c, c) for c in raw.columns]
    raw.index = pd.to_datetime(raw.index)
    if last_date not in raw.index:
        raise RuntimeError(f'新拉数据里没有锚点日 {last_date:%Y-%m-%d}, 无法安全拼接')
    new_dates = raw.index[raw.index > last_date]
    if not len(new_dates):
        print('没有新增交易日')
        return
    pct = raw.pct_change(fill_method=None)
    base = anchor.reindex(raw.columns)
    block = (1 + pct.loc[new_dates].fillna(0)).cumprod() * base           # 从锚点价复利滚算
    long = block.rename_axis(index='trade_date', columns='code').stack().dropna().rename('close').reset_index()
    long['trade_date'] = long['trade_date'].dt.date
    copy_in(long, 'stock_close_adj')
    print(f'新增 {len(new_dates)} 个交易日 ({new_dates[0]:%Y-%m-%d} ~ {new_dates[-1]:%Y-%m-%d})。{quota_str()}')

### 3.5 交易状态表

状态表必须和价格同步更新：状态表落后的那几天，可交易域是空的，02 回测会把这些尾部交易日截掉。

In [13]:
def update_state_table(end=None, dry_run=False):
    last_date = pd.Timestamp(scalar('SELECT max(trade_date) FROM stock_state'))
    jq = jq_auth()
    end_ts = pd.Timestamp(end) if end else last_closed_trade_day(jq)
    print(f'状态表最后一天 {last_date:%Y-%m-%d} → 目标 {end_ts:%Y-%m-%d}')
    if end_ts <= last_date:
        print('已是最新, 无需更新')
        return
    days = [pd.Timestamp(d) for d in jq.get_trade_days(
        start_date=str((last_date + pd.Timedelta(days=1)).date()), end_date=str(end_ts.date()))]
    print(f'需补 {len(days)} 个交易日')
    if dry_run or not days:
        return

    codes = read_sql('SELECT DISTINCT code FROM stock_state')['code'].tolist()
    jq_codes = [to_jq_code(c) for c in codes]
    back = dict(zip(jq_codes, codes))
    fields = ['close', 'paused', 'high_limit', 'low_limit']
    good, bad = isolate_bad_codes(jq, jq_codes, str(days[-1].date()), fields)
    if bad:
        print(f'{len(bad)} 只代码聚宽认不出, 已剔除: {[back.get(c, c) for c, _ in bad[:5]]}')
    info = jq.get_all_securities(types=['stock'], date=str(end_ts.date()))
    start_dates = {from_jq_code(i): pd.Timestamp(r['start_date']) for i, r in info.iterrows()}

    frames = []
    for d in days:
        ds = str(d.date())
        parts = []
        for i in range(0, len(good), BATCH_SIZE):
            try:
                px = jq.get_price(good[i:i + BATCH_SIZE], start_date=ds, end_date=ds, frequency='daily',
                                  fields=fields, skip_paused=False, fq=None, panel=False)
            except Exception as e:
                print(f'  ⚠ {ds} 批次取失败, 跳过: {e}')
                continue
            if px is not None and len(px):
                parts.append(px)
        if not parts:
            continue
        px = pd.concat(parts).set_index('code')
        try:
            st = jq.get_extras('is_st', good, start_date=ds, end_date=ds, df=True)
            st_row = st.iloc[0] if len(st) else pd.Series(dtype=bool)
        except Exception:
            st_row = pd.Series(dtype=bool)
        rows = []
        for j in px.index:
            code = back.get(j, from_jq_code(j))
            c, paused = px.at[j, 'close'], px.at[j, 'paused']
            hl, ll = px.at[j, 'high_limit'], px.at[j, 'low_limit']
            tradable = int((not bool(paused)) and pd.notna(c))
            mud = 0.0
            if pd.notna(c):
                if pd.notna(hl) and c >= hl:
                    mud = 1.0
                elif pd.notna(ll) and c <= ll:
                    mud = -1.0
            is_st = int(bool(st_row.get(j, False)))
            sd = start_dates.get(code)
            rows.append({'trade_date': d.date(), 'code': code,
                         'trade_status': '交易' if tradable else '停牌一天',   # 与原数据的中文取值一致
                         'status_t': tradable, 'maxupdown': mud,
                         'isst': is_st, 'issts': is_st,                       # 聚宽只给 is_st，两列填同一个值
                         'issusp': 0, 'lenlist': int((d - sd).days) if sd is not None else 0})
        frames.append(pd.DataFrame(rows))
    if frames:
        copy_in(pd.concat(frames, ignore_index=True), 'stock_state')

### 3.6 运行更新

默认会尝试更新；没有配置聚宽账号、没装 `jqdatasdk` 或 `HO_OFFLINE=1` 时自动跳过，只用库里已有的数据。

In [14]:
UPDATE = True     # 改成 False 可整体跳过联网更新
DRY_RUN = False   # True = 只打印计划

import importlib.util
can_update = (UPDATE and not OFFLINE and os.getenv('JQ_USER')
              and importlib.util.find_spec('jqdatasdk') is not None)
if not can_update:
    print('跳过联网更新（离线模式 / 未配置 JQ_USER / 未安装 jqdatasdk）')
else:
    print('=== 价格 ===')
    update_price_panel(dry_run=DRY_RUN)
    print('\n=== 状态表 ===')
    update_state_table(dry_run=DRY_RUN)
    print('\n=== 指数 ===')
    end = str(scalar('SELECT max(trade_date) FROM stock_close_adj'))
    for code, start in INDEX_START.items():
        update_index(code, start, end)
    for code in CONSTITUENT_INDEXES:
        if not scalar('SELECT EXISTS (SELECT 1 FROM index_constituents WHERE index_code = :c)', c=code):
            fetch_constituents(code, INDEX_START[code], end)
    print('\n' + quota_str())

overview()

跳过联网更新（离线模式 / 未配置 JQ_USER / 未安装 jqdatasdk）


,表,行数,起始,结束,代码数
0,stock_close_adj,"17,794,688",2000-01-04,2026-08-07,6106
1,stock_close_unadj,"16,111,553",2005-01-04,2026-08-07,5506
2,stock_state,"17,875,816",2000-01-04,2026-08-07,6108
3,index_close,"21,703",2005-01-04,2026-08-07,5
4,index_constituents,"7,284,239",2005-01-04,2026-07-31,6
